# Project 03 — Four Linear Algebra Methods

## Notebook 04 — Eigendecomposition

**Blocks:** 16–20  
**Dataset:** 400 MRI images; 128 × 128 grayscale; 16,384 features per image  
**Main project directory (Google Colab):** `MyDrive/MATHEMATICS_PROJECTS/PROJECT_03_LA_ML_FOUR_METHODS/`  
**Processed files:** `data/processed/X_mri_400.npy`, `y_mri_400.npy`, `filenames_mri_400.json`

*Execution note: Run cells in order after mounting Google Drive. Notebook 01 creates the processed dataset used by the following notebooks.*


### Mathematical purpose and interpretation

The Gram matrix $G=XX^{\mathsf T}$ is symmetric and positive semidefinite. Its eigenvalues equal the squared singular values of $X$ (including multiplicities), so its normalized eigenvalue sum quantifies the same **uncentered matrix energy** measured by SVD. This relationship allows the spectral structure to be analyzed using a $400\times400$ matrix rather than an explicit $16,384\times16,384$ feature-space matrix.

Selecting the leading **5, 44, 106, and 257** eigenvectors retains approximately **80.39%, 90.08%, 95.02%, and 99.01%** of Gram-matrix spectral energy. Reconstructed Gram-matrix errors are measured relative to $G$, **not** relative to the MRI image matrix $X$, so their numerical values should not be compared directly with the SVD image-matrix reconstruction errors. This notebook examines spectral properties, not diagnostic accuracy.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Block 16 — Load And Verify Mri Matrix


In [7]:

import numpy as np
from pathlib import Path

# BLOCK 16 — LOAD AND VERIFY MRI MATRIX

EXPECTED_SHAPE = (400, 16384)

path_1 = Path(
    "/content/drive/MyDrive/"
    "PROJECT_03_FOUR_LINEAR_ALGEBRA_METHODS_MRI/"
    "data/processed/X_mri_400.npy"
)

path_2 = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS/"
    "data/processed/X_mri_400.npy"
)

print("BLOCK 16 — LOAD AND VERIFY MRI MATRIX")

# Verify both files exist
assert path_1.is_file(), f"Missing file: {path_1}"
assert path_2.is_file(), f"Missing file: {path_2}"

# Load both matrices
X1 = np.load(path_1, allow_pickle=False)
X2 = np.load(path_2, allow_pickle=False)

# Compare numerical contents
same_shape = X1.shape == X2.shape
same_values = np.array_equal(X1, X2, equal_nan=True)

print("\nFILE COMPARISON")
print("Same dimensions:", same_shape)
print("Identical numerical contents:", same_values)

if not same_values:
    raise RuntimeError(
        "MRI matrices differ. Verify which file "
        "was used in the PCA/SVD notebooks."
    )

# Use the first verified matrix
X = X1.astype(np.float64)

n, p = X.shape

# Numerical verification
shape_ok = X.shape == EXPECTED_SHAPE
finite_ok = np.isfinite(X).all()
nonnegative_ok = np.all(X >= 0)
normalized_ok = np.all((X >= 0) & (X <= 1))

row_norms = np.linalg.norm(X, axis=1)
nonzero_rows_ok = np.all(row_norms > 0)

print("\nMATRIX DIMENSIONS")
print("X shape:", X.shape)
print("Number of MRI images:", n)
print("Features per image:", p)
print("Data type:", X.dtype)

print("\nNUMERICAL SUMMARY")
print("Minimum value:", X.min())
print("Maximum value:", X.max())
print("Mean value:", X.mean())
print("Standard deviation:", X.std())
print("Minimum row norm:", row_norms.min())
print("Maximum row norm:", row_norms.max())

print("\nFINAL VERIFICATION")
print("Expected dimensions:", shape_ok)
print("All values finite:", finite_ok)
print("All values nonnegative:", nonnegative_ok)
print("Values normalized to [0,1]:", normalized_ok)
print("All rows nonzero:", nonzero_rows_ok)

all_checks = all([
    same_shape,
    same_values,
    shape_ok,
    finite_ok,
    nonnegative_ok,
    normalized_ok,
    nonzero_rows_ok
])

print("\nALL BLOCK 16 CHECKS PASSED:", all_checks)

assert all_checks, "Block 16 verification failed."


BLOCK 16 — LOAD AND VERIFY MRI MATRIX

FILE COMPARISON
Same dimensions: True
Identical numerical contents: True

MATRIX DIMENSIONS
X shape: (400, 16384)
Number of MRI images: 400
Features per image: 16384
Data type: float64

NUMERICAL SUMMARY
Minimum value: 0.0
Maximum value: 1.0
Mean value: 0.18548444827339708
Standard deviation: 0.1962425453752799
Minimum row norm: 15.855797671509338
Maximum row norm: 71.4733338271951

FINAL VERIFICATION
Expected dimensions: True
All values finite: True
All values nonnegative: True
Values normalized to [0,1]: True
All rows nonzero: True

ALL BLOCK 16 CHECKS PASSED: True



### Block 16 — MRI Matrix Loading and Verification

The MRI dataset contains 400 images represented by
16,384 numerical features per observation.

- Matrix dimensions: (400, 16384)
- Data type: float64
- Pixel intensity range: [0, 1]
- Mean intensity: 0.18548445
- Standard deviation: 0.19624255

Two stored copies were compared and verified to
contain identical numerical values.

All numerical verification checks passed.

The matrix is ready for Gram matrix construction
and subsequent eigendecomposition.


## Block 17 — Gram Matrix Construction And Verification


In [8]:

import numpy as np

# BLOCK 17 — GRAM MATRIX CONSTRUCTION

# Verify that Block 16 produced the MRI matrix
assert "X" in globals(), "Run Block 16 first."
assert X.shape == (400, 16384)
assert np.isfinite(X).all()

# Construct Gram matrix
G = X @ X.T

# Matrix dimensions
n, m = G.shape

# Symmetry verification
symmetry_error = np.linalg.norm(G - G.T, ord="fro")
relative_symmetry_error = (
    symmetry_error / np.linalg.norm(G, ord="fro")
)

symmetry_ok = np.allclose(
    G, G.T, rtol=1e-10, atol=1e-10
)

# Eigendecomposition of symmetric Gram matrix
eigenvalues = np.linalg.eigvalsh(G)

# Positive semidefinite verification
scale = max(1.0, np.max(np.abs(eigenvalues)))
psd_tolerance = 1e-10 * scale

psd_ok = np.min(eigenvalues) >= -psd_tolerance

# Additional verification
shape_ok = G.shape == (400, 400)
finite_ok = np.isfinite(G).all()
diagonal_ok = np.allclose(
    np.diag(G),
    np.sum(X**2, axis=1),
    rtol=1e-10,
    atol=1e-10
)

print("BLOCK 17 — GRAM MATRIX CONSTRUCTION")

print("\nMATRIX DIMENSIONS")
print("X shape:", X.shape)
print("G shape:", G.shape)

print("\nGRAM MATRIX STATISTICS")
print("Minimum entry:", G.min())
print("Maximum entry:", G.max())
print("Trace:", np.trace(G))
print("Frobenius norm:", np.linalg.norm(G, ord="fro"))

print("\nSYMMETRY")
print("Symmetry error:", symmetry_error)
print("Relative symmetry error:", relative_symmetry_error)

print("\nEIGENVALUE SUMMARY")
print("Minimum eigenvalue:", eigenvalues.min())
print("Maximum eigenvalue:", eigenvalues.max())
print("PSD tolerance:", psd_tolerance)

print("\nFINAL VERIFICATION")
print("Correct dimensions:", shape_ok)
print("All values finite:", finite_ok)
print("Matrix symmetric:", symmetry_ok)
print("Positive semidefinite:", psd_ok)
print("Diagonal equals squared row norms:", diagonal_ok)

all_checks = all([
    shape_ok,
    finite_ok,
    symmetry_ok,
    psd_ok,
    diagonal_ok
])

print("\nALL BLOCK 17 CHECKS PASSED:", all_checks)


BLOCK 17 — GRAM MATRIX CONSTRUCTION

MATRIX DIMENSIONS
X shape: (400, 16384)
G shape: (400, 400)

GRAM MATRIX STATISTICS
Minimum entry: 163.89371191006362
Maximum entry: 5108.437448373664
Trace: 477859.788663391
Frobenius norm: 350309.06127656094

SYMMETRY
Symmetry error: 0.0
Relative symmetry error: 0.0

EIGENVALUE SUMMARY
Minimum eigenvalue: 0.0240546295932028
Maximum eigenvalue: 349460.0202906912
PSD tolerance: 3.494600202906912e-05

FINAL VERIFICATION
Correct dimensions: True
All values finite: True
Matrix symmetric: True
Positive semidefinite: True
Diagonal equals squared row norms: True

ALL BLOCK 17 CHECKS PASSED: True



### Block 17 — Interpretation of the Maximum Eigenvalue

**Question:** Why is the maximum eigenvalue so large (349,460.0203)?

**Explanation:**

The Gram matrix is constructed as G = XXᵀ, where X has
dimensions (400, 16384).

Although pixel intensities are normalized to [0, 1],
each Gram matrix entry involves products summed across
16,384 pixel features.

Therefore, large eigenvalues are mathematically plausible.

**Numerical Results:**

- Maximum eigenvalue: 349,460.0203
- Minimum eigenvalue: 0.024055
- Sum of eigenvalues (trace): 477,859.7887

**Dominant Spectral Energy:**

349,460.0203 / 477,859.7887 × 100 ≈ 73.13%

The largest eigenvalue accounts for approximately 73.13%
of the total spectral energy.

**Mathematical Interpretation:**

One dominant spectral direction captures approximately
73.13% of the total squared magnitude of the uncentered
MRI matrix.

This percentage represents spectral energy, not
necessarily statistical variance, because the matrix
has not been centered.

**Conclusion:**

The large maximum eigenvalue does not indicate a
computational error. Instead, it reveals a dominant
spectral direction in the MRI dataset.

This result motivates further eigenvalue and eigenvector
analysis in Block 18.


## Block 18 — Eigendecomposition And Eigenvalue Analysis


In [9]:

import numpy as np

# BLOCK 18 — EIGENDECOMPOSITION AND EIGENVALUE ANALYSIS

assert "G" in globals(), "Run Block 17 first."
assert G.shape == (400, 400)
assert np.allclose(G, G.T, rtol=1e-10, atol=1e-10)
assert np.isfinite(G).all()

# Eigendecomposition of the symmetric Gram matrix
eigenvalues, Q = np.linalg.eigh(G)

# Sort eigenvalues and eigenvectors in descending order
indices = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[indices]
Q = Q[:, indices]

# Diagonal eigenvalue matrix
Lambda = np.diag(eigenvalues)

# Reconstruction
G_reconstructed = Q @ Lambda @ Q.T

relative_reconstruction_error = (
    np.linalg.norm(G - G_reconstructed, ord="fro")
    / np.linalg.norm(G, ord="fro")
)

# Orthonormality
I = np.eye(Q.shape[1])

orthogonality_error = np.linalg.norm(
    Q.T @ Q - I, ord="fro"
)

# Eigenpair verification
eigenpair_error = (
    np.linalg.norm(G @ Q - Q * eigenvalues, ord="fro")
    / np.linalg.norm(G, ord="fro")
)

# Spectral energy
total_energy = np.sum(eigenvalues)

energy_ratio = eigenvalues / total_energy
cumulative_energy = np.cumsum(energy_ratio)

# Numerical checks
shape_ok = (
    Q.shape == (400, 400)
    and Lambda.shape == (400, 400)
)

reconstruction_ok = relative_reconstruction_error < 1e-10
orthogonality_ok = orthogonality_error < 1e-10
eigenpair_ok = eigenpair_error < 1e-10

scale = max(1.0, np.max(np.abs(eigenvalues)))
psd_ok = eigenvalues.min() >= -1e-10 * scale

energy_ok = np.isclose(
    np.sum(energy_ratio), 1.0, atol=1e-10
)

ordering_ok = np.all(np.diff(eigenvalues) <= 1e-10)

print("BLOCK 18 — EIGENDECOMPOSITION AND EIGENVALUE ANALYSIS")

print("\nMATRIX DIMENSIONS")
print("G shape:", G.shape)
print("Q shape:", Q.shape)
print("Lambda shape:", Lambda.shape)

print("\nEIGENVALUE SUMMARY")
print("Maximum eigenvalue:", eigenvalues[0])
print("Minimum eigenvalue:", eigenvalues[-1])
print("Sum of eigenvalues:", total_energy)

print("\nTOP 10 EIGENVALUES AND ENERGY")
for i in range(10):
    print(
        f"Eigenvalue {i+1:02d}: "
        f"{eigenvalues[i]:.6f} | "
        f"Energy: {100*energy_ratio[i]:.4f}% | "
        f"Cumulative: {100*cumulative_energy[i]:.4f}%"
    )

print("\nNUMERICAL ERRORS")
print("Relative reconstruction error:",
      relative_reconstruction_error)
print("Orthogonality error:", orthogonality_error)
print("Relative eigenpair error:", eigenpair_error)

print("\nFINAL VERIFICATION")
print("Correct dimensions:", shape_ok)
print("Reconstruction verified:", reconstruction_ok)
print("Eigenvectors orthonormal:", orthogonality_ok)
print("Eigenpairs verified:", eigenpair_ok)
print("Positive semidefinite:", psd_ok)
print("Spectral energy normalized:", energy_ok)
print("Eigenvalues descending:", ordering_ok)

all_checks = all([
    shape_ok,
    reconstruction_ok,
    orthogonality_ok,
    eigenpair_ok,
    psd_ok,
    energy_ok,
    ordering_ok
])

print("\nALL BLOCK 18 CHECKS PASSED:", all_checks)

assert all_checks, "Block 18 verification failed."


BLOCK 18 — EIGENDECOMPOSITION AND EIGENVALUE ANALYSIS

MATRIX DIMENSIONS
G shape: (400, 400)
Q shape: (400, 400)
Lambda shape: (400, 400)

EIGENVALUE SUMMARY
Maximum eigenvalue: 349460.0202906913
Minimum eigenvalue: 0.024054629593176658
Sum of eigenvalues: 477859.7886633912

TOP 10 EIGENVALUES AND ENERGY
Eigenvalue 01: 349460.020291 | Energy: 73.1302% | Cumulative: 73.1302%
Eigenvalue 02: 21206.349340 | Energy: 4.4378% | Cumulative: 77.5680%
Eigenvalue 03: 5620.434722 | Energy: 1.1762% | Cumulative: 78.7442%
Eigenvalue 04: 4997.692577 | Energy: 1.0458% | Cumulative: 79.7900%
Eigenvalue 05: 2864.208774 | Energy: 0.5994% | Cumulative: 80.3894%
Eigenvalue 06: 2760.528808 | Energy: 0.5777% | Cumulative: 80.9671%
Eigenvalue 07: 2560.118998 | Energy: 0.5357% | Cumulative: 81.5029%
Eigenvalue 08: 2308.561890 | Energy: 0.4831% | Cumulative: 81.9860%
Eigenvalue 09: 2108.866918 | Energy: 0.4413% | Cumulative: 82.4273%
Eigenvalue 10: 1896.368337 | Energy: 0.3968% | Cumulative: 82.8241%

NUMERICAL

## Block 19 — Component Selection By Spectral Energy


In [10]:
import numpy as np

# BLOCK 19 — COMPONENT SELECTION BY SPECTRAL ENERGY

assert "eigenvalues" in globals(), "Run Block 18 first."
assert eigenvalues.ndim == 1
assert len(eigenvalues) == 400

# Spectral energy
total_energy = np.sum(eigenvalues)
energy_ratio = eigenvalues / total_energy
cumulative_energy = np.cumsum(energy_ratio)

# Energy targets
targets = [0.80, 0.90, 0.95, 0.99]

selected_ranks = []
achieved_energies = []

for target in targets:
    k = np.searchsorted(cumulative_energy, target) + 1
    selected_ranks.append(k)
    achieved_energies.append(cumulative_energy[k - 1])

# Verification
energy_normalized_ok = np.isclose(np.sum(energy_ratio), 1.0, atol=1e-10)
cumulative_nondecreasing_ok = np.all(np.diff(cumulative_energy) >= -1e-12)
targets_reached_ok = all(a >= t for a, t in zip(achieved_energies, targets))
ranks_increasing_ok = np.all(np.diff(selected_ranks) > 0)

print("BLOCK 19 — COMPONENT SELECTION BY SPECTRAL ENERGY")

print("\nENERGY TARGETS")
for target, k, achieved in zip(targets, selected_ranks, achieved_energies):
    print(f"Target energy: {100*target:.0f}%")
    print("Required eigenvectors:", k)
    print(f"Achieved energy: {100*achieved:.4f}%")
    print()

print("FIRST 10 CUMULATIVE ENERGIES")
for i in range(10):
    print(f"k = {i+1:02d} | Cumulative energy: {100*cumulative_energy[i]:.4f}%")

print("\nFINAL VERIFICATION")
print("Energy is normalized:", energy_normalized_ok)
print("Cumulative energy is nondecreasing:", cumulative_nondecreasing_ok)
print("Energy targets reached:", targets_reached_ok)
print("Required ranks increase with targets:", ranks_increasing_ok)

all_checks = all([
    energy_normalized_ok,
    cumulative_nondecreasing_ok,
    targets_reached_ok,
    ranks_increasing_ok
])

print("\nALL BLOCK 19 CHECKS PASSED:", all_checks)

assert all_checks, "Block 19 verification failed."

BLOCK 19 — COMPONENT SELECTION BY SPECTRAL ENERGY

ENERGY TARGETS
Target energy: 80%
Required eigenvectors: 5
Achieved energy: 80.3894%

Target energy: 90%
Required eigenvectors: 44
Achieved energy: 90.0750%

Target energy: 95%
Required eigenvectors: 106
Achieved energy: 95.0181%

Target energy: 99%
Required eigenvectors: 257
Achieved energy: 99.0089%

FIRST 10 CUMULATIVE ENERGIES
k = 01 | Cumulative energy: 73.1302%
k = 02 | Cumulative energy: 77.5680%
k = 03 | Cumulative energy: 78.7442%
k = 04 | Cumulative energy: 79.7900%
k = 05 | Cumulative energy: 80.3894%
k = 06 | Cumulative energy: 80.9671%
k = 07 | Cumulative energy: 81.5029%
k = 08 | Cumulative energy: 81.9860%
k = 09 | Cumulative energy: 82.4273%
k = 10 | Cumulative energy: 82.8241%

FINAL VERIFICATION
Energy is normalized: True
Cumulative energy is nondecreasing: True
Energy targets reached: True
Required ranks increase with targets: True

ALL BLOCK 19 CHECKS PASSED: True


### Block 19 — Component Selection by Spectral Energy

Using the ordered eigenvalues of the Gram matrix, we
computed the cumulative spectral energy and determined
the minimum number of leading eigenvectors required to
reach selected energy targets.

For each target, we selected the smallest k such that

Cumulative Energy(k) >= target.

This analysis quantifies how many principal spectral
directions are needed to retain a desired proportion
of the total energy of the uncentered MRI dataset.

All numerical verification checks passed.

## Block 20 — Final Eigendecomposition Evaluation


In [11]:

import numpy as np

# BLOCK 20 — FINAL EIGENDECOMPOSITION EVALUATION

assert "G" in globals()
assert "Q" in globals()
assert "eigenvalues" in globals()
assert "selected_ranks" in globals()
assert "targets" in globals()

assert G.shape == (400, 400)
assert Q.shape == (400, 400)

gram_norm = np.linalg.norm(G, ord="fro")
squared_eigenvalues = eigenvalues**2

all_checks = True

print("BLOCK 20 — FINAL EIGENDECOMPOSITION EVALUATION")

for target, k in zip(targets, selected_ranks):

    # Leading eigenvectors and eigenvalues
    Q_k = Q[:, :k]
    lambda_k = eigenvalues[:k]

    # Low-rank Gram matrix reconstruction
    G_k = (Q_k * lambda_k) @ Q_k.T

    # Measured relative Frobenius error
    reconstruction_error = (
        np.linalg.norm(G - G_k, ord="fro")
        / gram_norm
    )

    # Theoretical relative Frobenius error
    theoretical_error = np.sqrt(
        np.sum(squared_eigenvalues[k:])
        / np.sum(squared_eigenvalues)
    )

    # Retained spectral energy
    retained_energy = (
        np.sum(lambda_k) / np.sum(eigenvalues)
    )

    # Verification
    error_ok = np.isclose(
        reconstruction_error,
        theoretical_error,
        rtol=1e-8,
        atol=1e-10
    )

    symmetry_ok = np.allclose(
        G_k, G_k.T,
        rtol=1e-10,
        atol=1e-10
    )

    energy_ok = retained_energy >= target - 1e-12

    all_checks = all_checks and all([
        error_ok,
        symmetry_ok,
        energy_ok
    ])

    print(f"\nTarget energy: {target*100:.0f}%")
    print("Required eigenvectors:", k)
    print("Reconstructed G shape:", G_k.shape)
    print(f"Retained energy: {retained_energy*100:.4f}%")
    print(f"Reconstruction error: {reconstruction_error:.8f}")
    print(f"Theoretical error: {theoretical_error:.8f}")
    print("Error agreement:", error_ok)
    print("Reconstruction symmetric:", symmetry_ok)
    print("Energy target reached:", energy_ok)

# Full reconstruction
G_full = (Q * eigenvalues) @ Q.T

full_error = (
    np.linalg.norm(G - G_full, ord="fro")
    / gram_norm
)

full_ok = full_error < 1e-10
all_checks = all_checks and full_ok

print("\nFULL EIGENDECOMPOSITION")
print("Full reconstruction error:", full_error)
print("Full reconstruction verified:", full_ok)

print("\nFINAL VERIFICATION")
print("ALL BLOCK 20 CHECKS PASSED:", all_checks)

assert all_checks, "Block 20 verification failed."


BLOCK 20 — FINAL EIGENDECOMPOSITION EVALUATION

Target energy: 80%
Required eigenvectors: 5
Reconstructed G shape: (400, 400)
Retained energy: 80.3894%
Reconstruction error: 0.02547716
Theoretical error: 0.02547716
Error agreement: True
Reconstruction symmetric: True
Energy target reached: True

Target energy: 90%
Required eigenvectors: 44
Reconstructed G shape: (400, 400)
Retained energy: 90.0750%
Reconstruction error: 0.01013558
Theoretical error: 0.01013558
Error agreement: True
Reconstruction symmetric: True
Energy target reached: True

Target energy: 95%
Required eigenvectors: 106
Reconstructed G shape: (400, 400)
Retained energy: 95.0181%
Reconstruction error: 0.00493045
Theoretical error: 0.00493045
Error agreement: True
Reconstruction symmetric: True
Energy target reached: True

Target energy: 99%
Required eigenvectors: 257
Reconstructed G shape: (400, 400)
Retained energy: 99.0089%
Reconstruction error: 0.00124674
Theoretical error: 0.00124674
Error agreement: True
Reconstruct


### Block 20 — Final Eigendecomposition Evaluation

The MRI Gram matrix was reconstructed using different
numbers of leading eigenvectors.

Numerical results:

- 80% spectral energy: 5 eigenvectors
  Reconstruction error: 0.02547716

- 90% spectral energy: 44 eigenvectors
  Reconstruction error: 0.01013558

- 95% spectral energy: 106 eigenvectors
  Reconstruction error: 0.00493045

- 99% spectral energy: 257 eigenvectors
  Reconstruction error: 0.00124674

Full eigendecomposition:
- Eigenvectors: 400
- Relative reconstruction error: 1.55e-15

Mathematical interpretation:

Increasing the number of eigenvectors reduces the
relative Frobenius reconstruction error.

The first five eigenvectors retain approximately
80.39% of the uncentered spectral energy while
producing a Gram matrix reconstruction error
of approximately 2.55%.

Spectral energy retention and Gram matrix reconstruction
error are different mathematical quantities.

The full eigendecomposition reconstructs the Gram matrix
with numerical precision close to machine accuracy.

Final verification:
- Theoretical and numerical errors agree.
- Reconstructed matrices are symmetric.
- All spectral energy targets were reached.
- Full reconstruction was verified.

ALL BLOCK 20 CHECKS PASSED: True
